### TTC Weekly Disruption Prediction Pipeline

In [3]:
import requests
import pandas as pd
from datetime import datetime
from pathlib import Path

In [9]:
# Toronto coordinates (weather forecast call)
TORONTO_LAT = 43.6532
TORONTO_LON = -79.3832

# GTFS raw files location
GTFS_DIR = Path("data/raw") 

# 50 selected TTC bus route short_names
SELECTED_ROUTES = [7, 11, 12, 24, 25, 29, 32, 34, 35, 36, 37, 38, 39, 40, 41, 43, 45, 47, 52, 53, 54, 59, 60, 
                   63, 68, 72, 73, 79, 84, 85, 86, 89, 94, 95, 96, 97, 100, 102, 108, 110, 116, 123, 129, 133, 165, 924, 925, 929, 939, 960]

In [ ]:
# Fetches an HOURLY 7-day weather forecast for Toronto using the Open-Meteo API
# Returns a DataFrame with one row per HOUR across the next 7 days
def get_weather_forecast():                     
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": TORONTO_LAT,
        "longitude": TORONTO_LON,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "wind_speed_10m",
            "visibility",
        ],
        "wind_speed_unit": "kmh",
        "timezone": "America/Toronto",
        "forecast_days": 7,
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    hourly = data["hourly"]
    weather_df = pd.DataFrame({
        "datetime": pd.to_datetime(hourly["time"]),
        "temp_celsius": hourly["temperature_2m"],
        "humidity_pct": hourly["relative_humidity_2m"],
        "precipitation_mm": hourly["precipitation"],
        "wind_speed_kmh": hourly["wind_speed_10m"],
        "visibility_km": [v / 1000 for v in hourly["visibility"]],  # Open-Meteo returns meters
    })

    weather_df["date"] = weather_df["datetime"].dt.date
    weather_df["hour"] = weather_df["datetime"].dt.hour

    return weather_df[[
        "datetime", "date", "hour",
        "temp_celsius", "humidity_pct", "precipitation_mm",
        "wind_speed_kmh", "visibility_km",
        "low_visibility", "high_wind",
    ]]

In [ ]:
# Loads the existing GTFS raw files and filters them down to the 50 selected TTC bus routes
# and merges them into a single trip-level DataFrame
def load_gtfs():
    routes = pd.read_csv(GTFS_DIR / "routes.txt")
    trips = pd.read_csv(GTFS_DIR / "trips.txt")
    stops = pd.read_csv(GTFS_DIR / "stops.txt")
    stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")
 
    # Filter routes down to the 50 selected route_short_names
    routes_filtered = routes[routes["route_short_name"].astype(str).isin(SELECTED_ROUTES)].copy()
 
    if routes_filtered.empty:
        raise ValueError(
            "No routes matched SELECTED_ROUTES against routes.txt's "
            "route_short_name column. Check that SELECTED_ROUTES contains "
            "the correct route numbers and that they're formatted the same "
            "way as in the GTFS file (e.g. '7' vs '07')."
        )
 
    # Keep only trips belonging to those routes
    trips_filtered = trips[trips["route_id"].isin(routes_filtered["route_id"])].copy()
 
    # Keep only stop_times belonging to those trips
    stop_times_filtered = stop_times[stop_times["trip_id"].isin(trips_filtered["trip_id"])].copy()
 
    # Build one row per trip: route_short_name, direction_id, and the
    # trip's scheduled starting hour (from its first stop_time) -- this
    # is what lets us know, per route/direction, what hours service runs
    first_stop_per_trip = (
        stop_times_filtered.sort_values("stop_sequence")
        .groupby("trip_id", as_index=False)
        .first()[["trip_id", "arrival_time"]]
    )
 
    trip_details = trips_filtered.merge(
        routes_filtered[["route_id", "route_short_name"]], on="route_id", how="left"
    ).merge(
        first_stop_per_trip, on="trip_id", how="left"
    )
 
    # GTFS times can exceed 24:00:00 (e.g. 25:30:00 for past-midnight
    # trips) -- parse the hour manually rather than with pd.to_datetime,
    # which would error on those values
    trip_details["scheduled_hour"] = (
        trip_details["arrival_time"]
        .str.split(":")
        .str[0]
        .astype(float)
        .mod(24)  # wrap 24+ hour values back into 0-23
        .astype("Int64")
    )
 
    return {
        "routes": routes_filtered,
        "trips": trips_filtered,
        "stop_times": stop_times_filtered,
        "stops": stops,
        "trip_details": trip_details,
    }

In [10]:
# ------------------------------------------------------------------
# Quick test — run this file directly to confirm it returns ~7 days
# of hourly data with all required columns
# ------------------------------------------------------------------
if __name__ == "__main__":
    weather = get_weather_forecast()

    print(weather.head(10))
    print(f"\nRows returned: {len(weather)}")
    print(f"Unique dates: {weather['date'].nunique()}")

    required_cols = [
        "temp_celsius", "humidity_pct", "precipitation_mm",
        "wind_speed_kmh", "visibility_km", "low_visibility", "high_wind",
    ]
    missing = [c for c in required_cols if c not in weather.columns]
    assert not missing, f"Missing expected columns: {missing}"

    assert weather["date"].nunique() == 7, (
        f"Expected 7 unique days, got {weather['date'].nunique()}"
    )
    print("PASSED: returned hourly forecast covering 7 days with all required columns.")

             datetime        date  hour  temp_celsius  humidity_pct  \
0 2026-07-15 00:00:00  2026-07-15     0          26.5            67   
1 2026-07-15 01:00:00  2026-07-15     1          26.1            69   
2 2026-07-15 02:00:00  2026-07-15     2          25.6            65   
3 2026-07-15 03:00:00  2026-07-15     3          25.4            68   
4 2026-07-15 04:00:00  2026-07-15     4          25.5            65   
5 2026-07-15 05:00:00  2026-07-15     5          24.4            68   
6 2026-07-15 06:00:00  2026-07-15     6          23.8            69   
7 2026-07-15 07:00:00  2026-07-15     7          24.1            71   
8 2026-07-15 08:00:00  2026-07-15     8          25.4            62   
9 2026-07-15 09:00:00  2026-07-15     9          25.0            52   

   precipitation_mm  wind_speed_kmh  visibility_km  low_visibility  high_wind  
0               0.0             9.3           25.6           False      False  
1               0.0             8.9           25.2        